### 1. 기본 환경 설정

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import torch

os.environ["HF_HUB_OFFLINE"] = "1"

candidates = [Path.cwd(), Path.cwd() / "query_router", Path("/data1/home/jonghak/AX-seoulfellow/query_router")]
ROUTER_DIR = next((p.resolve() for p in candidates if (p / "generate_vectors.py").is_file()), None)

if ROUTER_DIR is None:
    raise FileNotFoundError("query_router 폴더를 찾을 수 없습니다. ROUTER_DIR을 지정하세요.")

if str(ROUTER_DIR) not in sys.path:
    sys.path.insert(0, str(ROUTER_DIR))


from generate_vectors import TYPES, FEATURE_NAMES, load_encoder, encode, similarity_features, sha256
from train_linear import LinearRouter
from train_mlp import MLPRouter

# 다른 실험의 모델을 사용하려면 이 경로를 바꾸세요.
VECTORS_DIR = ROUTER_DIR / "artifacts/bge_m3"
CHECKPOINTS = {
    "linear": VECTORS_DIR / "linear/model.pt",
    "mlp": VECTORS_DIR / "mlp/model.pt",
}

DEVICE = "cpu"
torch.set_num_threads(4)
print("Python:", sys.executable)
print("실험 폴더:", VECTORS_DIR)


Python: /data1/home/jonghak/.conda/envs/router/bin/python
실험 폴더: /data1/home/jonghak/AX-seoulfellow/query_router/artifacts/bge_m3


### 2. 모델 불러오기

In [2]:
manifest_path = VECTORS_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_hash = sha256(manifest_path)
vectors_hash = sha256(VECTORS_DIR / "vectors.npz")

# Linear 모델과 MLP 모델은 명시적으로 각각 생성합니다.
models = {"linear": LinearRouter(), "mlp": MLPRouter()}
checkpoints = {}
for model_type, model in models.items():
    checkpoint = torch.load(CHECKPOINTS[model_type], map_location="cpu", weights_only=True)
    if checkpoint["model_type"] != model_type:
        raise ValueError(f"{model_type} 모델과 체크포인트의 종류가 다릅니다.")
    if checkpoint["manifest_sha256"] != manifest_hash or checkpoint["vectors_sha256"] != vectors_hash:
        raise ValueError(f"{model_type} 학습에 사용한 벡터/Encoder 설정과 다릅니다.")
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    checkpoints[model_type] = checkpoint
    print(f"{model_type.upper()}: {type(model).__name__} ← {CHECKPOINTS[model_type]}")

# 두 모델이 공유하는 Encoder와 온톨로지 벡터도 한 번만 로딩합니다.
encoder_config = dict(manifest["encoder"])
encoder_config["cache_dir"] = str(ROUTER_DIR / "artifacts/model_cache")
encoder = load_encoder(encoder_config, DEVICE)
with np.load(VECTORS_DIR / "vectors.npz", allow_pickle=False) as data:
    ontology_vectors = {kind: data[kind].copy() for kind in TYPES}


LINEAR: LinearRouter ← /data1/home/jonghak/AX-seoulfellow/query_router/artifacts/bge_m3/linear/model.pt
MLP: MLPRouter ← /data1/home/jonghak/AX-seoulfellow/query_router/artifacts/bge_m3/mlp/model.pt


/data1/home/jonghak/.conda/envs/router/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import json

jsonl_path = "/data1/home/jonghak/AX-seoulfellow/query_router/artifacts/bge_m3/test.jsonl"

testset_list = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        testset_list.append({
            "qa_id": data.get("qa_id"),
            "question": data.get("question")
        })

'수주 가능성을 높이기 위해 제안서 작성 전에 어떤 활동이 필요한가요?'

### 3. 질문 입력 및 두 모델 비교

In [15]:
query = testset_list[3]['question']

results = []
if not query.strip():
    print("query = ''의 따옴표 안에 질문을 입력하세욧.")
else:
    query_vector = encode(encoder, [query], encoder_config["query_prefix"], batch_size=1)
    features = similarity_features(query_vector, ontology_vectors)
    for model_type, model in models.items():
        checkpoint = checkpoints[model_type]
        
        normalized = (torch.from_numpy(features) - checkpoint["feature_mean"]) / checkpoint["feature_std"]
        with torch.no_grad():
            score = model(normalized).sigmoid().item()
        threshold = checkpoint["threshold"]
        results.append({
            "model": model_type.upper(),
            "needs_search": score >= threshold,
            "search_score": score,
            "threshold": threshold,
        })

    print("질문:", query)
    print()
    print(f"{'모델':<8} {'검색 필요 점수':<16} {'임계값':<10} 결과")
    print("-" * 62)
    for result in results:
        decision = "검색 필요" if result["needs_search"] else "검색 불필요"
        print(f"{result['model']:<10} {result['search_score']:<20.6f} {result['threshold']:<13.3f} {decision}")
        
    print("\n두 모델에 공통으로 입력된 최대 cos similarity:")
    for name, value in zip(FEATURE_NAMES, features[0]):
        print(f"- {name}: {float(value):.6f}")


질문: 수주영업에서 말하는 비즈니스 개발(B/D)은 무엇인가요?

모델       검색 필요 점수         임계값        결과
--------------------------------------------------------------
LINEAR     0.979972             0.500         검색 필요
MLP        0.999884             0.500         검색 필요

두 모델에 공통으로 입력된 최대 cos similarity:
- TaskFrame_max_cosine: 0.513522
- TaskMention_max_cosine: 0.530224
- Relation_max_cosine: 0.529027
